In [5]:
# Complete NLP Pipeline for Bank Review Analysis
# ------------------------------------------------

# Install if needed:
# pip install pandas nltk spacy transformers torch scikit-learn
# python -m spacy download en_core_web_sm

import pandas as pd
import nltk
import spacy

from nltk.corpus import stopwords
from transformers import pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

# ------------------------------------------------
# 1. Download NLTK resources
# ------------------------------------------------
nltk.download("stopwords")

# ------------------------------------------------
# 2. Load spaCy model
# ------------------------------------------------
nlp = spacy.load("en_core_web_sm")

# ------------------------------------------------
# 3. Load dataset
# ------------------------------------------------
df = pd.read_csv("cleaned_reviews.csv")

# Create review_id if missing
if "review_id" not in df.columns:
    df["review_id"] = range(1, len(df) + 1)

# ------------------------------------------------
# 4. Text Preprocessing Function
# ------------------------------------------------
stop_words = set(stopwords.words("english"))

def preprocess_text(text, lemmatize=True):
    
    if pd.isna(text):
        return ""
    
    # Lowercase
    text = str(text).lower()
    
    # spaCy processing
    doc = nlp(text)
    
    tokens = []
    
    for token in doc:
        
        # Remove stopwords, punctuation, spaces
        if token.is_stop:
            continue
        
        if token.is_punct or token.is_space:
            continue
        
        # Lemmatization
        if lemmatize:
            word = token.lemma_
        else:
            word = token.text
        
        tokens.append(word)
    
    return " ".join(tokens)

# Apply preprocessing
df["cleaned_review"] = df["review text"].apply(preprocess_text)

# ------------------------------------------------
# 5. Sentiment Analysis
# ------------------------------------------------
classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

def analyze_sentiment(text):
    
    result = classifier(text[:512])[0]
    
    label = result["label"].lower()
    score = result["score"]
    
    return pd.Series([label, score])

df[["sentiment_label", "sentiment_score"]] = (
    df["review text"].apply(analyze_sentiment)
)

# ------------------------------------------------
# 6. Theme Extraction using TF-IDF
# ------------------------------------------------
vectorizer = TfidfVectorizer(
    max_features=100,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(df["cleaned_review"])

terms = vectorizer.get_feature_names_out()

# Get top keyword for each review
def extract_theme(review_text):
    
    tfidf = vectorizer.transform([review_text])
    
    scores = tfidf.toarray()[0]
    
    if scores.sum() == 0:
        return "general feedback"
    
    top_index = scores.argmax()
    
    return terms[top_index]

df["identified_theme"] = df["cleaned_review"].apply(extract_theme)

# ------------------------------------------------
# 7. Final Output
# ------------------------------------------------
final_df = df[
    [
        "review_id",
        "review text",
        "sentiment_label",
        "sentiment_score",
        "identified_theme"
    ]
]

# Rename review column
final_df = final_df.rename(
    columns={"review text": "review_text"}
)

# ------------------------------------------------
# 8. Save Results
# ------------------------------------------------
final_df.to_csv("final_bank_review_analysis.csv", index=False)

print("\nPipeline completed successfully.")
print(final_df.head())

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bewha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 9370.73it/s]



Pipeline completed successfully.
   review_id                                        review_text  \
0          1  i love this bank and mobile banking of Comerci...   
1          2  ይህ አፕ ለምንድነው ሌላ ቀፎ ውስጥ ያለውን SIM ፕኒ ኮድ ስናስገባ የማ...   
2          3                                               good   
3          4  bad at processing, wht do u mean by I can't ma...   
4          5                                               good   

  sentiment_label  sentiment_score identified_theme  
0        positive         0.999764             bank  
1        negative         0.965139              cbe  
2        positive         0.999816             good  
3        negative         0.999664          payment  
4        positive         0.999816             good  
